In [4]:
from scservo_sdk import sms_sts, PortHandler

print("sms_sts methods available:")
print("  WheelMode:", hasattr(sms_sts,'WheelMode'))
print("  WriteSpec:", hasattr(sms_sts, 'WriteSpec'))
print("  WritePosEx:", hasattr(sms_sts, 'WritePosEx'))

sms_sts methods available:
  WheelMode: True
  WriteSpec: True
  WritePosEx: True


In [6]:
import serial.tools.list_ports

ports = serial.tools.list_ports.comports()
print("Available serial ports: ")
for port in ports:
    print(f"  {port.device}")
    print(f"    Description: {port.description}")
    print(f"    Manufacturer: {port.manufacturer}")
    print()

Available serial ports: 
  COM7
    Description: Standard Serial over Bluetooth link (COM7)
    Manufacturer: Microsoft

  COM5
    Description: Standard Serial over Bluetooth link (COM5)
    Manufacturer: Microsoft

  COM4
    Description: Standard Serial over Bluetooth link (COM4)
    Manufacturer: Microsoft

  COM6
    Description: Standard Serial over Bluetooth link (COM6)
    Manufacturer: Microsoft

  COM11
    Description: USB-Enhanced-SERIAL CH343 (COM11)
    Manufacturer: wch.cn



In [7]:
SERIAL_PORT = 'COM11'
print(f"Using serial port: {SERIAL_PORT}")

Using serial port: COM11


In [38]:
from scservo_sdk import sms_sts, PortHandler

# Configuration
BAUDRATE = 1000000
SERVO_ID = 1
port_handler = PortHandler(SERIAL_PORT)
servo = sms_sts(port_handler)


if port_handler.ser is not None:
    try:
        port_handler.ser.close()
    except Exception:
        pass
        
try:
    port_handler.closePort()
except:
    print("No port already open")
# Open the serial port
if port_handler.openPort():
    print("✓ Port opened successfully")
else:
    print("✗ Failed to open port")

# Set the baud rate
if port_handler.setBaudRate(BAUDRATE):
    print(f"✓ Baud rate set to {BAUDRATE}")
else:
    print("✗ Failed to set baud rate")
model_number, comm_result, error = servo.ping(SERVO_ID)

if comm_result == 0:  # COMM_SUCCESS
    print(f"✓ Servo ID {SERVO_ID} found!")
    print(f"  Model number: {model_number}")
else:
    print(f"✗ Failed to ping servo")

for test_id in range(1, 11):
    model, comm_result, _ = servo.ping(test_id)
    if comm_result == 0:
        print(f"✓ Detected servo at ID: {test_id} (Model: {model})")
        break
else:
    print("✗ No STS servo responded on IDs 1–10 at 1 Mbps")

No port already open
✓ Port opened successfully
✓ Baud rate set to 1000000
✓ Servo ID 1 found!
  Model number: 777
✓ Detected servo at ID: 1 (Model: 777)


In [46]:
ADDR_PRESENT_POSITION = 56
# Read current position (2 bytes)
position, comm_result, error = servo.read2ByteTxRx(SERVO_ID, ADDR_PRESENT_POSITION)
if comm_result == 0:
    degrees = position * 360 / 4096  # Convert to degrees
    print(f"Current Position: {position} (raw) = {degrees:.1f}°")
else:
    print(f"Failed to read position")

Current Position: 4542 (raw) = 399.2°


In [47]:
ADDR_PRESENT_VOLTAGE = 62
# Read voltage (1 byte)
voltage_raw, comm_result, error = servo.read1ByteTxRx(SERVO_ID, ADDR_PRESENT_VOLTAGE)
if comm_result == 0:
    voltage = voltage_raw * 0.1  # Convert to volts
    print(f"Voltage: {voltage:.1f}V")
else:
    print(f"Failed to read voltage")

Voltage: 5.1V


In [48]:
ADDR_PRESENT_TEMPERATURE = 63
temp, comm_result, error = servo.read1ByteTxRx(SERVO_ID, ADDR_PRESENT_TEMPERATURE)
if comm_result == 0:
    print(f"Temperature: {temp}°C")
else:
    print(f"Failed to read temperature")

Temperature: 31°C


In [49]:
ADDR_TORQUE_ENABLE = 40
# Read torque enable status (1 byte)
torque_enabled, comm_result, error = servo.read1ByteTxRx(SERVO_ID, ADDR_TORQUE_ENABLE)
if comm_result == 0:
    print(f"Torque Enabled: {'Yes' if torque_enabled else 'No'}")
else:
    print(f"Failed to read torque status")

Torque Enabled: No


In [50]:
ADDR_MOVING = 66
# Read moving status (1 byte)
moving, comm_result, error = servo.read1ByteTxRx(SERVO_ID, ADDR_MOVING)
if comm_result == 0:
    print(f"Moving: {'Yes' if moving else 'No'}")
else:
    print(f"Failed to read moving status")

Moving: No


In [59]:
ADDR_TORQUE_ENABLE = 40
ADDR_GOAL_POSITION = 42

# Enable torque
servo.write1ByteTxRx(SERVO_ID, ADDR_TORQUE_ENABLE, 1)
print("✓ Torque enabled - servo is now holding position")

# Move to position (2048 = 180°)
move_to = 1000
degrees = move_to * 360 / 4096  # Convert to degrees
# WritePosEx(ID, Position, Speed, Acceleration)
servo.WritePosEx(SERVO_ID, move_to, 1000, 50)
print(f"✓ Moving to position ({move_to} = {degrees}°)")

✓ Torque enabled - servo is now holding position
✓ Moving to position (1000 = 87.890625°)


In [66]:
# Move slowly with smooth acceleration
# WritePosEx(ID, Position, Speed, Acceleration)
servo.WritePosEx(SERVO_ID, 1024, 1000, 0)  # Slow, smooth
print("✓ Moving slowly to 90°")

import time
time.sleep(3)

# Move quickly with instant acceleration
servo.WritePosEx(SERVO_ID, 3072, 2000, 100)  # Fast, instant
print("✓ Moving quickly to 270°")

✓ Moving slowly to 90°
✓ Moving quickly to 270°


In [72]:
import time

# Register addresses
ADDR_PRESENT_POSITION = 56
ADDR_PRESENT_LOAD = 60
ADDR_PRESENT_CURRENT = 69
ADDR_MOVING = 66

def read_servo_status():
    """Read and display current servo status"""
    # Read position
    position, _, _ = servo.read2ByteTxRx(SERVO_ID, ADDR_PRESENT_POSITION)
    degrees = position * 360 / 4096
    
    # Read load (bits 0-9 = magnitude, bit 10 = direction)
    load_raw, _, _ = servo.read2ByteTxRx(SERVO_ID, ADDR_PRESENT_LOAD)
    load_magnitude = load_raw & 0x3FF  # Lower 10 bits
    load_direction = "CW" if load_raw & 0x400 else "CCW"  # Bit 10
    load_percent = load_magnitude / 1000 * 100
    
    # Read current draw
    current_raw, _, _ = servo.read2ByteTxRx(SERVO_ID, ADDR_PRESENT_CURRENT)
    current_ma = current_raw * 6.5  # Convert to milliamps
    
    # Read moving status
    moving, _, _ = servo.read1ByteTxRx(SERVO_ID, ADDR_MOVING)
    
    print(f"Position: {degrees:6.1f}° | Load: {load_percent:5.1f}% {load_direction} | Current: {current_ma:6.1f}mA | Moving: {'Yes' if moving else 'No'}")

# Move to a position and monitor
print("Moving servo and monitoring status...\n")

# Start a move
servo.WritePosEx(SERVO_ID, 0, 1000, 50)

# Monitor for 2 seconds
for i in range(20):
    read_servo_status()
    time.sleep(0.1)

print("\nDone!")

Moving servo and monitoring status...

Position:  179.5° | Load:   3.2% CW | Current:    6.5mA | Moving: Yes
Position:  178.9° | Load:  16.0% CCW | Current:   19.5mA | Moving: Yes
Position:  172.1° | Load:  22.4% CCW | Current:   26.0mA | Moving: Yes
Position:  162.9° | Load:  20.0% CCW | Current:   19.5mA | Moving: Yes
Position:  153.9° | Load:  20.8% CCW | Current:   26.0mA | Moving: Yes
Position:  144.9° | Load:  22.8% CCW | Current:   39.0mA | Moving: Yes
Position:  135.9° | Load:  23.2% CCW | Current:   45.5mA | Moving: Yes
Position:  126.8° | Load:  24.4% CCW | Current:   52.0mA | Moving: Yes
Position:  117.6° | Load:  24.4% CCW | Current:   39.0mA | Moving: Yes
Position:  108.6° | Load:  26.0% CCW | Current:   45.5mA | Moving: Yes
Position:   99.6° | Load:  26.0% CCW | Current:   52.0mA | Moving: Yes
Position:   90.5° | Load:  26.8% CCW | Current:   58.5mA | Moving: Yes
Position:   81.5° | Load:  27.2% CCW | Current:   58.5mA | Moving: Yes
Position:   72.3° | Load:  27.2% CCW | 

In [73]:
# Register addresses
ADDR_SERVO_STATUS = 65
ADDR_PRESENT_VOLTAGE = 62
ADDR_PRESENT_TEMPERATURE = 63
ADDR_MAX_TEMP_LIMIT = 13
ADDR_MAX_VOLTAGE_LIMIT = 14
ADDR_MIN_VOLTAGE_LIMIT = 15

# Error status bitmask definitions
ERROR_FLAGS = {
    0: "Voltage Error",
    1: "Sensor Error", 
    2: "Temperature Error",
    3: "Current Error",
    5: "Overload Error",
}

# Read servo status register
status, comm_result, error = servo.read1ByteTxRx(SERVO_ID, ADDR_SERVO_STATUS)

if comm_result == 0:
    print("=== Servo Error Status ===\n")
    
    if status == 0:
        print("✓ No errors - servo is healthy!")
    else:
        print(f"⚠ Status byte: {status} (binary: {bin(status)})\n")
        print("Active errors:")
        for bit, name in ERROR_FLAGS.items():
            if status & (1 << bit):
                print(f"  ✗ {name}")
else:
    print(f"Failed to read status")

# Read current conditions for context
print("\n=== Current Conditions ===\n")

voltage_raw, _, _ = servo.read1ByteTxRx(SERVO_ID, ADDR_PRESENT_VOLTAGE)
temp, _, _ = servo.read1ByteTxRx(SERVO_ID, ADDR_PRESENT_TEMPERATURE)
max_temp, _, _ = servo.read1ByteTxRx(SERVO_ID, ADDR_MAX_TEMP_LIMIT)
max_volt, _, _ = servo.read1ByteTxRx(SERVO_ID, ADDR_MAX_VOLTAGE_LIMIT)
min_volt, _, _ = servo.read1ByteTxRx(SERVO_ID, ADDR_MIN_VOLTAGE_LIMIT)

print(f"Voltage:     {voltage_raw * 0.1:.1f}V  (limits: {min_volt * 0.1:.1f}V - {max_volt * 0.1:.1f}V)")
print(f"Temperature: {temp}°C  (limit: {max_temp}°C)")

=== Servo Error Status ===

✓ No errors - servo is healthy!

=== Current Conditions ===

Voltage:     5.1V  (limits: 4.0V - 12.0V)
Temperature: 32°C  (limit: 80°C)


In [76]:
ADDR_TORQUE_ENABLE = 40

# Disable torque - servo will go limp
servo.write1ByteTxRx(SERVO_ID, ADDR_TORQUE_ENABLE, 0)
print("✓ Torque disabled")

# Close the port
port_handler.closePort()
print("✓ Port closed")

✓ Torque disabled
✓ Port closed


In [77]:
from scservo_sdk import sms_sts, PortHandler

# Reconnect to servo
# SERIAL_PORT = '/dev/cu.usbmodemXXXXX'  # Replace with your port
BAUDRATE = 1000000
SERVO_ID = 1

port_handler = PortHandler(SERIAL_PORT)
servo = sms_sts(port_handler)

port_handler.openPort()
port_handler.setBaudRate(BAUDRATE)
print("✓ Connected to servo")

# Enable wheel mode
servo.WheelMode(SERVO_ID)
print("✓ Wheel mode enabled")

# Spin using WriteSpec(ID, Speed, Acceleration)
# Speed: positive = one direction, negative = other direction
speed = 500
acceleration = 50
servo.WriteSpec(SERVO_ID, speed, acceleration)
print(f"✓ Spinning at speed {speed}")

✓ Connected to servo
✓ Wheel mode enabled
✓ Spinning at speed 500


In [78]:
import time

# Change direction - use negative speed for opposite direction
print("--- Spinning CCW (negative speed) ---")
servo.WriteSpec(SERVO_ID, -500, 50)
time.sleep(2)

# Speed up
print("--- Faster CW ---")
servo.WriteSpec(SERVO_ID, 1500, 50)
time.sleep(2)

# Slow down
print("--- Slower CW ---")
servo.WriteSpec(SERVO_ID, 200, 50)
time.sleep(2)

# Stop the motor
print("--- Stopping ---")
servo.WriteSpec(SERVO_ID, 0, 50)
print("✓ Motor stopped")

--- Spinning CCW (negative speed) ---
--- Faster CW ---
--- Slower CW ---
--- Stopping ---
✓ Motor stopped


In [79]:
# Register addresses
ADDR_MIN_ANGLE_LIMIT = 9
ADDR_MAX_ANGLE_LIMIT = 11
ADDR_MODE = 33
ADDR_TORQUE_ENABLE = 40

# Disable torque to change EEPROM settings
servo.write1ByteTxRx(SERVO_ID, ADDR_TORQUE_ENABLE, 0)
print("✓ Torque disabled")

# Restore angle limits (0-4095 = full 360° range)
servo.write2ByteTxRx(SERVO_ID, ADDR_MIN_ANGLE_LIMIT, 0)
servo.write2ByteTxRx(SERVO_ID, ADDR_MAX_ANGLE_LIMIT, 4095)
print("✓ Angle limits restored (0-4095)")

# Set mode back to 0 (Position servo mode)
servo.write1ByteTxRx(SERVO_ID, ADDR_MODE, 0)
print("✓ Mode set to 0 (position control)")

# Enable torque
servo.write1ByteTxRx(SERVO_ID, ADDR_TORQUE_ENABLE, 1)
print("✓ Torque enabled")

# Test by moving to center position
servo.WritePosEx(SERVO_ID, 2048, 1000, 50)
print("✓ Moving to center position (2048)")

✓ Torque disabled
✓ Angle limits restored (0-4095)
✓ Mode set to 0 (position control)
✓ Torque enabled
✓ Moving to center position (2048)


In [80]:
ADDR_TORQUE_ENABLE = 40

# Disable torque and close port
servo.write1ByteTxRx(SERVO_ID, ADDR_TORQUE_ENABLE, 0)
print("✓ Torque disabled")

port_handler.closePort()
print("✓ Port closed")

print("\n" + "="*50)
print("TUTORIAL COMPLETE!")
print("="*50)

✓ Torque disabled
✓ Port closed

TUTORIAL COMPLETE!
